# 🎬 Video AI Backend - Google Colab

هذا الكود يشغل خادم Flask لاستقبال طلبات توليد الفيديو من تطبيق المخيم الرقمي.

## المميزات:
- ✅ مجاني 100% باستخدام GPU من Google
- ✅ لا يحتاج GPU محلي
- ✅ يستخدم AnimateDiff لتوليد فيديو
- ✅ يعمل طالما الكولاب مشغل

## الخطوة 1: تثبيت المكتبات (اضغط ▶️ وانتظر 2 دقيقة)

In [ ]:
!pip install -q flask flask-cors pyngrok torch torchvision torchaudio
!pip install -q diffusers transformers accelerate
!pip install -q moviepy
print("✅ تم تثبيت المكتبات")

## الخطوة 2: إعداد ngrok

1. اذهب لـ https://dashboard.ngrok.com/get-started/your-authtoken
2. انسخ الـ Authtoken
3. ضعه في الكود أدناه بدلاً من YOUR_TOKEN_HERE

In [ ]:
from pyngrok import ngrok

# ✏️ ضع توكنك هنا:
NGROK_AUTH_TOKEN = "YOUR_TOKEN_HERE"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✅ تم إعداد ngrok")

## الخطوة 3: تحميل موديل توليد الفيديو (انتظر 3-5 دقائق)

In [ ]:
import torch
from diffusers import AnimateDiffPipeline, DDIMScheduler
from PIL import Image
import os

# إنشاء مجلد للفيديوهات
os.makedirs("generated_videos", exist_ok=True)

print("🔄 جاري تحميل الموديل... انتظر")

# تحميل AnimateDiff
model_id = "guoyww/animatediff-motion-adapter-v1-5-2"
pipe = AnimateDiffPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# إضافة Motion Adapter
from diffusers import MotionAdapter
adapter = MotionAdapter.from_pretrained(
    "guoyww/animatediff-motion-adapter-v1-5-2",
    torch_dtype=torch.float16
)

pipe.unet.enable_forward_chunking(chunk_size=1, dim=1)

print("✅ تم تحميل الموديل بنجاح!")

## الخطوة 4: تشغيل الخادم (هيطلع لك الرابط هنا)

In [ ]:
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
import uuid
import numpy as np
from moviepy.editor import ImageSequenceClip
import threading

app = Flask(__name__)
CORS(app)

generation_status = {}

@app.route('/')
def home():
    return jsonify({
        "status": "running",
        "service": "Video AI Generator",
        "gpu": str(torch.cuda.get_device_name(0)) if torch.cuda.is_available() else "CPU"
    })

@app.route('/generate', methods=['POST'])
def generate():
    try:
        data = request.json
        prompt = data.get('prompt', '')
        submission_id = data.get('submission_id', str(uuid.uuid4()))
        
        if not prompt:
            return jsonify({"error": "Prompt required"}), 400
        
        generation_status[submission_id] = {"status": "generating"}
        print(f"🎬 توليد: {prompt[:50]}...")
        
        # توليد الفيديو
        output = pipe(
            prompt=prompt,
            num_frames=16,
            guidance_scale=7.5,
            num_inference_steps=25
        )
        
        # حفظ الفيديو
        frames = [np.array(img) for img in output.frames[0]]
        clip = ImageSequenceClip(frames, fps=8)
        video_path = f"generated_videos/{submission_id}.mp4"
        clip.write_videofile(video_path, codec='libx264', audio=False, verbose=False)
        
        # رفع الفيديو على خدمة خارجية (اختياري) أو حفظه محليا
        # للتبسيط: نرجع رابط مباشر
        video_url = f"/video/{submission_id}"
        
        generation_status[submission_id] = {
            "status": "completed",
            "video_url": video_url
        }
        
        print(f"✅ تم: {video_url}")
        return jsonify({"success": True, "video_url": video_url, "submission_id": submission_id})
        
    except Exception as e:
        print(f"❌ خطأ: {e}")
        return jsonify({"error": str(e)}), 500

@app.route('/video/<submission_id>')
def serve_video(submission_id):
    video_path = f"generated_videos/{submission_id}.mp4"
    if os.path.exists(video_path):
        return send_file(video_path, mimetype='video/mp4')
    return jsonify({"error": "Not found"}), 404

# تشغيل ngrok والخادم
print("🚀 تشغيل الخادم...")
public_url = ngrok.connect(5000, "http")
print("\n" + "="*60)
print(f"✅ الخادم شغال!")
print(f"🔗 الرابط: {public_url.public_url}")
print("="*60 + "\n")
print("📋 انسخ الرابط ده وروح للتطبيق → مسابقة الفيديو → ⚙️ الإعدادات")

app.run(host='0.0.0.0', port=5000)

## 📋 ملخص

1. شغل الخلايا بالترتيب (1→2→3→4)
2. انتظر تحميل الموديل (3-5 دقائق)
3. انسخ رابط ngrok
4. الصقه في التطبيق
5. ⚠️ **خلي صفحة Colab مفتوحة!**